# Boosting Cardiac Risk Prediction
### A Hybrid Machine Learning Framework with Stacked Gradient Boosting and Bayesian Hyperparameter Tuning

**Mohammad Nadeem, Parupalli Bhargav Kumar**
Undergraduate Research Project (2025), School of Computer Science Engineering and Technology, Bennett University
Supervised by Dr. Shailza Kanwar

This notebook implements the full pipeline:

1. Data acquisition (UCI Heart Disease, Cleveland data)
2. Exploratory data analysis
3. Train/test split and outlier removal (z-score + IQR)
4. Pre-processing: imputation, one-hot encoding, standard scaling
5. Feature selection: Pearson correlation check + SelectKBest (ANOVA F)
6. Class balancing with Borderline-SMOTE
7. Base learners (XGBoost, LightGBM, CatBoost, HistGradientBoosting) tuned with Optuna
8. Stacking ensemble, baselines and evaluation (accuracy, precision, recall, F1, ROC-AUC, confusion matrix, ROC curves)

**How to run:** open in Google Colab and choose *Runtime → Run all*. If Colab asks to restart the session after the install cell, restart and choose *Run all* again. With the default settings it takes roughly 10–20 minutes on a free Colab CPU. All figures, tables and the trained model are written to the `results/` folder and downloaded as `results.zip` at the end.

In [ ]:
# Libraries that are not installed in Colab by default
!pip install -q -U xgboost lightgbm catboost optuna imbalanced-learn ucimlrepo

In [ ]:
import os, json, time, shutil, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             roc_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report)

import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import optuna
import imblearn
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import BorderlineSMOTE
import joblib

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid')

SEED = 42          # fixed random seed for reproducibility
TEST_SIZE = 0.20   # stratified 80:20 train-test split
CV_FOLDS = 5       # stratified k-fold cross-validation
N_TRIALS = 40      # Optuna trials per base learner (increase for a more thorough search)
OUT = 'results'
os.makedirs(OUT, exist_ok=True)
np.random.seed(SEED)

def save(name):
    plt.tight_layout()
    plt.savefig(f'{OUT}/{name}.png', dpi=300, bbox_inches='tight')
    plt.show()

print('Python', platform.python_version())
for m in [np, pd, sklearn, xgb, lgb, cb, optuna, imblearn]:
    print(f'{m.__name__:12s}{m.__version__}')

## 1. Data acquisition

We use the **UCI Heart Disease dataset (Cleveland data)**: 303 patients, 13 clinical attributes and the diagnosis `num` (0 = no heart disease, 1–4 = heart disease present). Following common practice, `num` is converted to a binary target (0 = no disease, 1 = disease).

| Attribute | Description |
|---|---|
| age | age in years |
| sex | 1 = male, 0 = female |
| cp | chest pain type (1 typical angina, 2 atypical angina, 3 non-anginal pain, 4 asymptomatic) |
| trestbps | resting blood pressure (mm Hg) |
| chol | serum cholesterol (mg/dl) |
| fbs | fasting blood sugar > 120 mg/dl (1 = true) |
| restecg | resting ECG results (0 normal, 1 ST-T abnormality, 2 left ventricular hypertrophy) |
| thalach | maximum heart rate achieved |
| exang | exercise-induced angina (1 = yes) |
| oldpeak | ST depression induced by exercise relative to rest |
| slope | slope of the peak exercise ST segment (1 up, 2 flat, 3 down) |
| ca | number of major vessels (0–3) coloured by fluoroscopy |
| thal | thalassemia (3 normal, 6 fixed defect, 7 reversible defect) |

*Note:* the 1,025-row `heart.csv` file often shared on Kaggle is built from the same patients and contains many duplicated records. Duplicates end up in both the training and test sets and inflate the test accuracy, so the original 303-record data is used here.

In [ ]:
COLS = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
        'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
URL = ('https://archive.ics.uci.edu/ml/machine-learning-databases/'
       'heart-disease/processed.cleveland.data')

def load_cleveland():
    # 1) official UCI package
    try:
        from ucimlrepo import fetch_ucirepo
        heart = fetch_ucirepo(id=45)
        data = pd.concat([heart.data.features[COLS], heart.data.targets[['num']]], axis=1)
        print('Loaded with ucimlrepo')
        return data
    except Exception as e:
        print('ucimlrepo not available:', e)
    # 2) raw file from the UCI archive
    try:
        data = pd.read_csv(URL, header=None, names=COLS + ['num'], na_values='?')
        print('Loaded from the UCI archive')
        return data
    except Exception as e:
        print('Download failed:', e)
    # 3) local copy: upload processed.cleveland.data to the Colab Files panel
    data = pd.read_csv('processed.cleveland.data', header=None, names=COLS + ['num'], na_values='?')
    print('Loaded from local file')
    return data

df = load_cleveland()
df[COLS] = df[COLS].apply(pd.to_numeric, errors='coerce')
df['target'] = (df['num'] > 0).astype(int)   # 0 = no heart disease, 1 = heart disease
df = df.drop(columns='num').reset_index(drop=True)
print('Shape:', df.shape)
df.head()

In [ ]:
missing = df.isna().sum()
print('Missing values:', missing[missing > 0].to_dict())

counts = df['target'].value_counts().sort_index()
N_NEG, N_POS = int(counts[0]), int(counts[1])
print(f'No heart disease (0): {N_NEG}   Heart disease (1): {N_POS}   '
      f'ratio = {N_POS / N_NEG:.2f}')
df.describe().T.round(2)

## 2. Exploratory data analysis

In [ ]:
CONT = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
CAT = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x='target', hue='target', palette='Set2', legend=False, ax=ax)
ax.set_xticks([0, 1], ['No disease', 'Disease'])
ax.set_xlabel('')
ax.set_title('Class distribution')
save('class_distribution')

plt.figure(figsize=(11, 9))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True,
            cbar_kws={'shrink': 0.8})
plt.title('Pearson correlation heatmap')
save('correlation_heatmap')

In [ ]:
fig, axes = plt.subplots(2, len(CONT), figsize=(20, 8))
for i, col in enumerate(CONT):
    sns.histplot(data=df, x=col, hue='target', kde=True, palette='Set2', ax=axes[0, i])
    sns.boxplot(data=df, x='target', y=col, hue='target', palette='Set2', legend=False, ax=axes[1, i])
    axes[1, i].set_xticks([0, 1], ['No disease', 'Disease'])
    axes[1, i].set_xlabel('')
plt.suptitle('Continuous attributes by class', y=1.01)
save('continuous_distributions')

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, col in zip(axes.ravel(), CAT):
    sns.countplot(data=df, x=col, hue='target', palette='Set2', ax=ax)
    ax.set_title(col)
plt.suptitle('Categorical attributes by class', y=1.01)
save('categorical_distributions')

g = sns.pairplot(df[CONT + ['target']], hue='target', palette='Set2', corner=True,
                 plot_kws={'s': 15, 'alpha': 0.7})
g.savefig(f'{OUT}/pairplot.png', dpi=200)
plt.show()

## 3. Train/test split and outlier removal

The data are split first (stratified 80:20), so the test set is never used for any decision. Outliers are then removed **from the training set only**, using two methods on the continuous attributes:

* **z-score:** |z| > 3
* **IQR:** value outside [Q1 − 1.5·IQR, Q3 + 1.5·IQR]

A record is removed only if both methods flag it, which keeps the removal conservative on a small medical dataset.

In [ ]:
X = df.drop(columns='target')
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)

def outlier_mask(data, cols, z_thr=3.0, iqr_k=1.5):
    z_flag = pd.Series(False, index=data.index)
    iqr_flag = pd.Series(False, index=data.index)
    for c in cols:
        x = data[c]
        z = (x - x.mean()) / x.std()
        z_flag |= z.abs() > z_thr
        q1, q3 = x.quantile(0.25), x.quantile(0.75)
        iqr = q3 - q1
        iqr_flag |= (x < q1 - iqr_k * iqr) | (x > q3 + iqr_k * iqr)
    return z_flag & iqr_flag, z_flag, iqr_flag

mask, z_flag, iqr_flag = outlier_mask(X_train, CONT)
N_OUTLIERS = int(mask.sum())
print(f'Flagged by z-score: {int(z_flag.sum())}   by IQR: {int(iqr_flag.sum())}   '
      f'removed (both): {N_OUTLIERS}')
print(X_train[mask][CONT])

X_train, y_train = X_train[~mask], y_train[~mask]
print('\nTraining set:', X_train.shape, y_train.value_counts().sort_index().to_dict())
print('Test set:    ', X_test.shape, y_test.value_counts().sort_index().to_dict())

## 4. Pre-processing

* `ca` and `thal` contain a few missing values: median imputation for numeric attributes, most-frequent imputation for nominal ones.
* Nominal attributes (`cp`, `restecg`, `slope`, `thal`) are one-hot encoded; the other attributes are already numeric (binary or ordinal).
* All numeric attributes are standardized with `StandardScaler`.

These steps are part of every model pipeline, so they are fitted on the training folds only (no information from validation or test data leaks into training).

In [ ]:
NUM = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca', 'sex', 'fbs', 'exang']
NOM = ['cp', 'restecg', 'slope', 'thal']
# all possible codes from the UCI documentation, so every fold produces the same columns
NOM_CATEGORIES = [[1., 2., 3., 4.], [0., 1., 2.], [1., 2., 3.], [3., 6., 7.]]

def make_preprocessor():
    numeric = Pipeline([('impute', SimpleImputer(strategy='median')),
                        ('scale', StandardScaler())])
    nominal = Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                        ('onehot', OneHotEncoder(categories=NOM_CATEGORIES, handle_unknown='ignore',
                                                          sparse_output=False))])
    return ColumnTransformer([('num', numeric, NUM), ('nom', nominal, NOM)])

def clean_names(names):
    return [n.split('__', 1)[1] for n in names]

pre = make_preprocessor().fit(X_train)
FEATURES = clean_names(pre.get_feature_names_out())
N_FEATURES = len(FEATURES)
print(f'{N_FEATURES} features after encoding:')
print(FEATURES)

## 5. Feature selection

1. **Pearson correlation** on the encoded training data, to check for highly correlated (redundant) features.
2. **SelectKBest with the ANOVA F-statistic** to rank features by how strongly they separate the two classes. The number of retained features *k* is tuned by Optuna together with each model's hyperparameters (Section 7).

In [ ]:
Xt = pd.DataFrame(pre.transform(X_train), columns=FEATURES, index=X_train.index)

corr = Xt.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack().sort_values(ascending=False)
print('Most correlated feature pairs (|r|):')
print(pairs.head(8).round(3).to_string())
print('Pairs with |r| > 0.8:', int((pairs > 0.8).sum()))

F, p = f_classif(Xt, y_train)
anova = (pd.DataFrame({'feature': FEATURES, 'F': F, 'p_value': p})
           .sort_values('F', ascending=False).reset_index(drop=True))
plt.figure(figsize=(8, 7))
sns.barplot(data=anova, y='feature', x='F', color='steelblue')
plt.title('ANOVA F-scores (training data)')
save('anova_f_scores')
anova.round(4)

## 6. Model pipeline and base learners

Every model is trained inside the same pipeline:

**pre-processing → SelectKBest (ANOVA F, k) → Borderline-SMOTE → classifier**

The pipeline comes from `imbalanced-learn`, so Borderline-SMOTE is applied only while fitting, i.e. only to training data. Validation folds and the test set are never oversampled. Borderline-SMOTE creates synthetic minority samples close to the class boundary, where misclassification is most likely.

The four gradient boosting libraries are wrapped in one small scikit-learn compatible class so that they behave identically inside pipelines, cross-validation and the stacking ensemble.

In [ ]:
class Booster(ClassifierMixin, BaseEstimator):
    # Common wrapper for XGBoost, LightGBM, CatBoost and HistGradientBoosting
    def __init__(self, kind='xgb', params=None, random_state=SEED):
        self.kind = kind
        self.params = params
        self.random_state = random_state

    def _build(self):
        p = dict(self.params or {})
        if self.kind == 'xgb':
            return xgb.XGBClassifier(**p, random_state=self.random_state, n_jobs=1,
                                     eval_metric='logloss', verbosity=0)
        if self.kind == 'lgbm':
            return lgb.LGBMClassifier(**p, random_state=self.random_state, n_jobs=1, verbose=-1)
        if self.kind == 'cat':
            return cb.CatBoostClassifier(**p, random_seed=self.random_state, thread_count=1,
                                         verbose=0, allow_writing_files=False)
        if self.kind == 'hgb':
            return HistGradientBoostingClassifier(**p, random_state=self.random_state)
        raise ValueError(f'unknown model kind: {self.kind}')

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y)
        self.classes_ = np.unique(y)
        self.model_ = self._build()
        self.model_.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(np.asarray(X, dtype=float))

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


def make_pipeline(model, k='all'):
    return ImbPipeline([
        ('pre', make_preprocessor()),
        ('select', SelectKBest(f_classif, k=k)),
        ('smote', BorderlineSMOTE(random_state=SEED)),
        ('model', model),
    ])

MODELS = {'XGBoost': 'xgb', 'LightGBM': 'lgbm', 'CatBoost': 'cat', 'HistGradientBoosting': 'hgb'}
CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

# quick sanity check: every base learner trains with default settings
for name, kind in MODELS.items():
    s = cross_val_score(make_pipeline(Booster(kind)), X_train, y_train, cv=CV, scoring='roc_auc')
    print(f'{name:22s} default CV ROC-AUC = {s.mean():.4f} ± {s.std():.4f}')

## 7. Bayesian hyperparameter optimization with Optuna

For each base learner, Optuna's Tree-structured Parzen Estimator (a Bayesian optimization method) searches the hyperparameters and the number of selected features *k*. The objective is the **mean 5-fold stratified cross-validated ROC-AUC on the training set**; the test set is not used.

In [ ]:
def suggest_params(trial, kind):
    lr = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    if kind == 'xgb':
        return {'n_estimators': trial.suggest_int('n_estimators', 100, 600, step=50),
                'max_depth': trial.suggest_int('max_depth', 2, 6),
                'learning_rate': lr,
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True)}
    if kind == 'lgbm':
        return {'n_estimators': trial.suggest_int('n_estimators', 100, 600, step=50),
                'num_leaves': trial.suggest_int('num_leaves', 4, 32),
                'max_depth': trial.suggest_int('max_depth', 2, 8),
                'learning_rate': lr,
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 40),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'subsample_freq': 1,
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True)}
    if kind == 'cat':
        return {'iterations': trial.suggest_int('iterations', 200, 800, step=100),
                'depth': trial.suggest_int('depth', 3, 7),
                'learning_rate': lr,
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True)}
    if kind == 'hgb':
        return {'max_iter': trial.suggest_int('max_iter', 100, 500, step=50),
                'max_depth': trial.suggest_int('max_depth', 2, 8),
                'learning_rate': lr,
                'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 8, 31),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 40),
                'l2_regularization': trial.suggest_float('l2_regularization', 1e-3, 10.0, log=True)}
    raise ValueError(kind)


def tune(kind):
    def objective(trial):
        params = suggest_params(trial, kind)
        k = trial.suggest_int('k', 6, N_FEATURES)
        trial.set_user_attr('params', params)
        pipe = make_pipeline(Booster(kind, params), k=k)
        return cross_val_score(pipe, X_train, y_train, cv=CV, scoring='roc_auc', n_jobs=-1).mean()

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    return study


studies, best = {}, {}
for name, kind in MODELS.items():
    t0 = time.time()
    study = tune(kind)
    studies[name] = study
    best[name] = {'kind': kind,
                  'k': study.best_params['k'],
                  'params': study.best_trial.user_attrs['params'],
                  'best_cv_roc_auc': study.best_value}
    print(f'{name:22s} best CV ROC-AUC = {study.best_value:.4f}   k = {best[name]["k"]}'
          f'   ({time.time() - t0:.0f} s)')

with open(f'{OUT}/best_hyperparameters.json', 'w') as f:
    json.dump(best, f, indent=2)

In [ ]:
plt.figure(figsize=(7, 4.5))
for name, study in studies.items():
    values = [t.value for t in study.trials if t.value is not None]
    plt.plot(range(1, len(values) + 1), np.maximum.accumulate(values), label=name)
plt.xlabel('Trial')
plt.ylabel('Best CV ROC-AUC so far')
plt.title('Optuna optimization history')
plt.legend()
save('optuna_history')

for name, b in best.items():
    print(f'\n{name}  (k = {b["k"]})')
    for p, v in b['params'].items():
        print(f'   {p:20s} {v:.4g}' if isinstance(v, float) else f'   {p:20s} {v}')

## 8. Stacking ensemble, baselines and evaluation

The four tuned pipelines are combined with **stacked generalization**: the base learners' out-of-fold predicted probabilities (5-fold) are used to train a **logistic regression meta-learner**, which makes the final prediction.

For comparison, two simple baselines (logistic regression and random forest) are trained with the same pre-processing, feature selection and Borderline-SMOTE.

Each model is evaluated on the held-out test set. We also report the 5-fold cross-validated ROC-AUC on the training set (mean ± standard deviation, with a different fold split from the one used for tuning), which is more stable than a single test split on a small dataset.

In [ ]:
def tuned_pipeline(name):
    b = best[name]
    return make_pipeline(Booster(b['kind'], b['params']), k=b['k'])

final_models = {name: tuned_pipeline(name) for name in MODELS}

stack = StackingClassifier(
    estimators=[(name, tuned_pipeline(name)) for name in MODELS],
    final_estimator=LogisticRegression(max_iter=1000),
    stack_method='predict_proba',
    cv=StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED),
    n_jobs=1)

baselines = {
    'Logistic Regression': make_pipeline(LogisticRegression(max_iter=1000)),
    'Random Forest': make_pipeline(RandomForestClassifier(n_estimators=300, random_state=SEED)),
}
all_models = {**baselines, **final_models, 'Stacked Ensemble': stack}

EVAL_CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED + 1)
rows, probas = [], {}
for name, model in all_models.items():
    t0 = time.time()
    cv_auc = cross_val_score(model, X_train, y_train, cv=EVAL_CV, scoring='roc_auc', n_jobs=-1)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    probas[name] = proba
    rows.append({'Model': name,
                 'Accuracy': accuracy_score(y_test, pred),
                 'Precision': precision_score(y_test, pred, zero_division=0),
                 'Recall': recall_score(y_test, pred),
                 'F1-score': f1_score(y_test, pred),
                 'ROC-AUC': roc_auc_score(y_test, proba),
                 'CV ROC-AUC mean': cv_auc.mean(),
                 'CV ROC-AUC sd': cv_auc.std()})
    print(f'{name:22s} done ({time.time() - t0:.0f} s)')

results = pd.DataFrame(rows).set_index('Model')
results.to_csv(f'{OUT}/results_table.csv')
results.round(4)

In [ ]:
# Table in the format used in the paper
table = results[['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC']].apply(
    lambda col: col.map('{:.3f}'.format))
table['CV ROC-AUC (mean ± SD)'] = [f'{m:.3f} ± {s:.3f}' for m, s in
                                   zip(results['CV ROC-AUC mean'], results['CV ROC-AUC sd'])]
table.to_csv(f'{OUT}/results_table_formatted.csv')
print(table.to_string())

best_single = results.loc[list(MODELS)].sort_values('ROC-AUC', ascending=False).index[0]
s, b = results.loc['Stacked Ensemble'], results.loc[best_single]
print(f'\nStacked ensemble: accuracy {s["Accuracy"]:.3f}, ROC-AUC {s["ROC-AUC"]:.4f}')
print(f'Best single base learner ({best_single}): accuracy {b["Accuracy"]:.3f}, ROC-AUC {b["ROC-AUC"]:.4f}')
print(f'Difference: accuracy {100 * (s["Accuracy"] - b["Accuracy"]):+.1f} percentage points, '
      f'ROC-AUC {s["ROC-AUC"] - b["ROC-AUC"]:+.4f}')

In [ ]:
pred_stack = (probas['Stacked Ensemble'] >= 0.5).astype(int)

fig, ax = plt.subplots(figsize=(4.8, 4.2))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_stack),
                       display_labels=['No disease', 'Disease']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Stacked ensemble (test set)')
ax.grid(False)
save('confusion_matrix_stacked')
print(classification_report(y_test, pred_stack, target_names=['No disease', 'Disease'], digits=3))

plt.figure(figsize=(6.5, 5.5))
for name, proba in probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    strong = name == 'Stacked Ensemble'
    plt.plot(fpr, tpr, lw=2.5 if strong else 1.2, ls='-' if strong or name in MODELS else ':',
             label=f'{name} (AUC = {roc_auc_score(y_test, proba):.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=0.8)
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curves (test set)')
plt.legend(loc='lower right', fontsize=8)
save('roc_curves')

In [ ]:
# Features kept by SelectKBest in each tuned base learner
selected = {}
for name in MODELS:
    pipe = final_models[name]
    names = np.array(clean_names(pipe.named_steps['pre'].get_feature_names_out()))
    selected[name] = names[pipe.named_steps['select'].get_support()].tolist()
    print(f'{name} (k = {len(selected[name])}): {", ".join(selected[name])}')

# Weight the meta-learner gives to each base learner
weights = pd.Series(stack.final_estimator_.coef_[0], index=list(MODELS))
print('\nMeta-learner (logistic regression) coefficients:')
print(weights.round(3).to_string())

## 9. Summary of the experimental setup

In [ ]:
summary = {
    'dataset': 'UCI Heart Disease (Cleveland), 303 records, 13 attributes, binary target',
    'class_counts': {'no_disease': N_NEG, 'disease': N_POS},
    'missing_values': {k: int(v) for k, v in missing[missing > 0].items()},
    'split': f'stratified {int((1 - TEST_SIZE) * 100)}:{int(TEST_SIZE * 100)}, seed {SEED}',
    'outliers_removed_from_training_set': N_OUTLIERS,
    'training_records_after_cleaning': int(len(X_train)),
    'test_records': int(len(X_test)),
    'features_after_encoding': N_FEATURES,
    'cv_folds': CV_FOLDS,
    'optuna_trials_per_model': N_TRIALS,
    'optuna_objective': f'mean {CV_FOLDS}-fold stratified CV ROC-AUC (training set)',
    'selected_k': {name: best[name]['k'] for name in MODELS},
    'selected_features': selected,
    'meta_learner': 'Logistic regression on base learners\' out-of-fold probabilities',
    'best_hyperparameters': {name: best[name]['params'] for name in MODELS},
    'test_results': results.round(4).to_dict(orient='index'),
    'library_versions': {m.__name__: m.__version__ for m in [np, pd, sklearn, xgb, lgb, cb, optuna, imblearn]},
}
with open(f'{OUT}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

for key in ['dataset', 'class_counts', 'missing_values', 'split', 'outliers_removed_from_training_set',
            'training_records_after_cleaning', 'test_records', 'features_after_encoding', 'cv_folds',
            'optuna_trials_per_model', 'optuna_objective', 'selected_k', 'meta_learner']:
    print(f'{key:36s} {summary[key]}')

## 10. Save the model and predict a new patient

The saved file contains the complete pipeline (imputation, encoding, scaling, feature selection and the stacked ensemble), so it takes the 13 raw clinical attributes as input. To load it in another notebook or a Streamlit app, define the `Booster` class first and then call `joblib.load`.

In [ ]:
joblib.dump(stack, f'{OUT}/stacked_model.joblib')

patient = pd.DataFrame([{'age': 63, 'sex': 1, 'cp': 1, 'trestbps': 145, 'chol': 233, 'fbs': 1,
                         'restecg': 2, 'thalach': 150, 'exang': 0, 'oldpeak': 2.3, 'slope': 3,
                         'ca': 0, 'thal': 6}])
prob = stack.predict_proba(patient)[0, 1]
print(f'Predicted probability of heart disease: {prob:.3f} -> '
      f'{"heart disease" if prob >= 0.5 else "no heart disease"}')

In [ ]:
shutil.make_archive('results', 'zip', OUT)
print('Saved files:', sorted(os.listdir(OUT)))
try:
    from google.colab import files
    files.download('results.zip')
except ImportError:
    print('results.zip is in the current working directory')